In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
#Extracion de datos
movies_df = spark.read.parquet(f"{silver_folder_path}/movies")
movie_genre_df = spark.read.parquet(f"{silver_folder_path}/movie_genre")
genre_df = spark.read.parquet(f"{silver_folder_path}/genre")

movies_df.show(4)
movie_genre_df.show(4)
genre_df.show(4)

In [0]:
#Tabla de movies filtrada con los campos y datos que necesitamos

movies_df = movies_df.filter(
                               (col("year_release_date") >= 2015)
                             )\
                    .select(movies_df.movie_id,
                            movies_df.year_release_date, 
                            movies_df.budget,
                            movies_df.revenue
                            )
movies_df.show(4)

In [0]:
#eneramos la tabla agregada con los campos solicitados
genre_movie_df = genre_df.join( movie_genre_df,
                                genre_df.genre_id == movie_genre_df.genre_id,
                                "inner"
                               )\
                          .select(movie_genre_df.movie_id, genre_df.genre_name)


movies_genre_final_df = movies_df.join(genre_movie_df,
                                       movies_df.movie_id == genre_movie_df.movie_id,
                                       "inner"
                                       )\
                                 .select(movies_df["*"], genre_movie_df.genre_name)


In [0]:

movies_genre_agg_df = movies_genre_final_df.groupBy("year_release_date", "genre_name")\
                                           .agg(
                                                 sum("budget").alias("budget"),
                                                 sum("revenue").alias("revenue")   
                                               )

movies_genre_agg_df.show(3)                                                    

In [0]:
#"movies_genre_agg_df.select("year_release_date")
result_group_movie_genre_df = movies_genre_agg_df.select( "year_release_date","genre_name", "budget", "revenue")\
                                         .withColumn("dense_rank", dense_rank().over(Window.partitionBy("year_release_date")
                                                                               .orderBy(desc("budget"))
                                                                               .orderBy(desc("revenue"))
                                                                         )
                                                     )
                                        
result_group_movie_genre_df.display()

In [0]:

result_group_movie_genre_df = add_ingestion_date(result_group_movie_genre_df)
result_group_movie_genre_df = add_env(result_group_movie_genre_df)

In [0]:
#Guardamos en la capa gold 
result_group_movie_genre_df.write.mode("overwrite").format("parquet").save(f"{gold_folder_path}/result_group_movie_genre_df")

df = spark.read.parquet(f"{gold_folder_path}/result_group_movie_genre_df")
display(df)
